# 🏭 AI Project 2: Support Vector Classification (SVC)
## Kathmandu Valley Air Quality Index (AQI) Classification

### Objective:
Classify hourly air pollution health risk levels in Kathmandu Valley into EPA/Nepal AQI standards (`Good`, `Moderate`, `Unhealthy for Sensitive Groups`, `Unhealthy`, `Hazardous`) using Support Vector Machines (SVC).

### SVC Mathematical Foundation:
Support Vector Classifiers find the maximum-margin hyperplane separating classes:

$$\min_{w, b, \xi} \frac{1}{2} \|w\|^2 + C \sum_{i=1}^n \xi_i$$
$$\text{subject to } y_i (w^T \phi(x_i) + b) \ge 1 - \xi_i, \quad \xi_i \ge 0$$

Where:
- $\phi(x)$ maps samples to a higher dimensional space using kernels like RBF ($K(x, x') = \exp(-\gamma \|x-x'\|^2)$).
- $C$ is the penalty parameter for misclassifications (trade-off between margin width and classification error).
- Multi-class classification is handled via One-vs-Rest (OvR) or One-vs-One (OvO) decision boundaries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

sns.set_theme(style="whitegrid")

## 1. Load and Inspect Kathmandu Air Quality Dataset

In [ ]:
df = pd.read_csv('data/kathmandu_air_quality.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
print("AQI Target Distribution:")
print(df['AQI_Category'].value_counts())

## 2. Exploratory Data Analysis (Pollution in Nepal)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='Season', y='Raw Conc.', palette='Set2', ax=axes[0])
axes[0].set_title('Kathmandu PM2.5 Concentration by Season', fontweight='bold')
axes[0].set_ylabel('PM2.5 (µg/m³)')

sns.countplot(data=df, x='AQI_Category', palette='YlOrRd', ax=axes[1])
axes[1].set_title('Frequency of AQI Severity Categories', fontweight='bold')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 3. Data Preprocessing & Train/Test Split

In [ ]:
num_features = [
    'Raw Conc.', 'NowCast Conc.', 'PM25_Lag1', 'PM25_Lag3', 'PM25_Roll6h',
    'Hour_Sin', 'Hour_Cos', 'Month_Sin', 'Month_Cos'
]
cat_features = ['Season']

X = df[num_features + cat_features]
y = df['AQI_Category'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_features)
])

## 4. Train SVC with Different Kernels (Linear, RBF, Poly)

In [ ]:
for kernel in ['linear', 'rbf', 'poly']:
    model = Pipeline([
        ('prep', preprocessor),
        ('svc', SVC(kernel=kernel, C=10.0, class_weight='balanced', random_state=42))
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average='macro')
    print(f"Kernel: {kernel.upper():<7} | Accuracy: {acc*100:.2f}% | Macro F1: {f1:.4f}")

## 5. Hyperparameter Tuning (GridSearchCV)

In [ ]:
param_grid = {
    'svc__C': [1.0, 10.0, 50.0],
    'svc__gamma': ['scale', 0.05, 0.1]
}

grid = GridSearchCV(
    Pipeline([('prep', preprocessor), ('svc', SVC(kernel='rbf', class_weight='balanced', random_state=42))]),
    param_grid, cv=3, scoring='f1_macro', n_jobs=-1
)
grid.fit(X_train, y_train)
print("Best Parameters:", grid.best_params_)
best_svc = grid.best_estimator_

## 6. Evaluation & Confusion Matrix

In [ ]:
y_pred = best_svc.predict(X_test)
print(f"Overall Test Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

labels = ['Good', 'Moderate', 'Unhealthy_Sensitive', 'Unhealthy', 'Hazardous']
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd', xticklabels=labels, yticklabels=labels)
plt.title('Kathmandu AQI Classification Confusion Matrix', fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 7. Interactive AQI & Health Advisory Predictor

In [ ]:
def check_air_quality(pm25_val, month, hour, season):
    sample = pd.DataFrame([{
        'Raw Conc.': pm25_val,
        'NowCast Conc.': pm25_val,
        'PM25_Lag1': pm25_val * 0.95,
        'PM25_Lag3': pm25_val * 0.9,
        'PM25_Roll6h': pm25_val * 0.92,
        'Hour_Sin': np.sin(2 * np.pi * hour / 24.0),
        'Hour_Cos': np.cos(2 * np.pi * hour / 24.0),
        'Month_Sin': np.sin(2 * np.pi * month / 12.0),
        'Month_Cos': np.cos(2 * np.pi * month / 12.0),
        'Season': season
    }])
    pred = best_svc.predict(sample)[0]
    print(f"PM2.5: {pm25_val} µg/m³ in {season} ({hour}:00) -> Predicted AQI Category: {pred}")

check_air_quality(pm25_val=110.0, month=1, hour=9, season='Winter')
check_air_quality(pm25_val=12.0, month=7, hour=15, season='Monsoon')